# Notebook 06 — County & Market Recommendation System

**Objective:** Recommend counties with lowest food security risk and best agricultural conditions using cosine similarity on weather-IPC feature vectors.

**Approach:** Content-based filtering

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import section, COUNTIES

master = pd.read_csv("data/processed/master_dataset.csv")
ipc    = pd.read_csv("data/processed/ipc_clean.csv")
print("Data loaded. Master shape:", master.shape)

## Step 1 — Build County Feature Profiles

In [ ]:
section("COUNTY PROFILES — Weather + Food Security Features")

# Aggregate to one row per county
county_profile_cols = [
    "total_rainfall", "mean_temp", "mean_solar",
    "mean_humidity", "dry_days", "spi_3"
]
available = [c for c in county_profile_cols if c in master.columns]

county_profiles = (
    master.groupby("county")[available]
    .mean()
    .reset_index()
)

# Add IPC phase (worst phase per county)
ipc_county = (
    ipc.groupby("county")["ipc_phase"]
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={"ipc_phase": "ipc_phase"})
)
county_profiles = county_profiles.merge(ipc_county, on="county", how="left")

# Composite food security score (lower IPC = better, higher rainfall = better)
county_profiles["food_security_score"] = (
    (4 - county_profiles["ipc_phase"].fillna(3)) * 0.5 +
    county_profiles["spi_3"].fillna(0).clip(-2, 2) * 0.3 +
    county_profiles["total_rainfall"].fillna(0) / 1000 * 0.2
)

print("County profiles:")
county_profiles.sort_values("food_security_score", ascending=False).head(10)

## Step 2 — Cosine Similarity Recommendation

In [ ]:
section("RECOMMENDATION — Top Counties by Similarity")

# Normalise features for cosine similarity
scaler = MinMaxScaler()
feature_cols = [c for c in available if c in county_profiles.columns]
X_profiles = scaler.fit_transform(county_profiles[feature_cols].fillna(0))

# Compute similarity matrix
sim_matrix = cosine_similarity(X_profiles)
sim_df = pd.DataFrame(sim_matrix,
                      index=county_profiles["county"],
                      columns=county_profiles["county"])

def recommend_similar_counties(target_county: str, top_n: int = 5) -> pd.DataFrame:
    if target_county not in sim_df.index:
        print(f"County '{target_county}' not found")
        return pd.DataFrame()
    similar = sim_df[target_county].sort_values(ascending=False)[1:top_n+1]
    return similar.reset_index().rename(columns={target_county: "similarity_score"})

# Example: find counties similar to Turkana (high risk)
print("Counties most similar to Turkana (for resource allocation):")
print(recommend_similar_counties("Turkana"))

print("\nCounties most similar to Kiambu (for best practice replication):")
print(recommend_similar_counties("Kiambu"))

## Step 3 — Top Counties Ranking

In [ ]:
section("RANKING — Counties by Food Security Score")

ranking = county_profiles.sort_values("food_security_score", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colours = ["#EF5350" if row["ipc_phase"] == 3
           else "#FFA726" if row["ipc_phase"] == 2
           else "#66BB6A"
           for _, row in ranking.iterrows()]

ax.barh(ranking["county"], ranking["food_security_score"], color=colours, alpha=0.85)
ax.set_title("Kenya Counties — Food Security Score\n(Green=Minimal, Orange=Stressed, Red=Crisis)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Composite Food Security Score (higher = better)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("reports/figures/10_county_ranking.png", dpi=150, bbox_inches="tight")
plt.show()